# Multi-Model Comparison — AI Code Review Assistant

This notebook compares evaluation results across multiple LLM models to determine
the best model for the AI Code Review Assistant.

## Setup

Run evaluations for each model first:
```bash
# DeepSeek V4 Flash (via NVIDIA NIM)
python -m evaluation.evaluate --mode security --rate-limit 2.0

# GPT-4o-mini (via OpenAI)
python -m evaluation.evaluate --model gpt-4o-mini --base-url https://api.openai.com/v1 --rate-limit 1.0

# Llama 3.1 8B (via NVIDIA NIM)
python -m evaluation.evaluate --model meta/llama-3.1-8b-instruct --rate-limit 2.0
```

Then compute metrics for each:
```bash
python -m evaluation.metrics evaluation/results/run_<model>_<timestamp>.json
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))

try:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.rcParams['figure.figsize'] = (14, 7)
    matplotlib.rcParams['font.size'] = 12
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print('matplotlib not installed — charts skipped')

try:
    import numpy as np
    HAS_NP = True
except ImportError:
    HAS_NP = False

from evaluation.metrics import compute_metrics, load_run

## 1. Load All Model Results

In [ ]:
results_dir = Path('evaluation/results')

# Find all run files (exclude metrics/report files)
run_files = sorted(results_dir.glob('run_*.json'))
run_files = [f for f in run_files if '_metrics' not in f.name and '_report' not in f.name]

print(f'Found {len(run_files)} evaluation run(s):')
for f in run_files:
    print(f'  - {f.name}')

# Load and compute metrics for each
model_metrics = {}
model_runs = {}
for f in run_files:
    run_data = load_run(str(f))
    # Skip runs with errors in first result (broken runs)
    first = run_data['results'][0]
    if first.get('summary', {}).get('error'):
        print(f'  SKIPPING {f.name} (has errors: {first["summary"]["error"][:60]}...)')
        continue
    metrics = compute_metrics(run_data)
    model_name = metrics['model']
    model_metrics[model_name] = metrics
    model_runs[model_name] = run_data
    print(f'  Loaded: {model_name} (F1={metrics["f1_score"]:.1%})')

if not model_metrics:
    raise ValueError('No valid evaluation runs found. Run evaluations first.')

models = list(model_metrics.keys())
print(f'\nModels for comparison: {models}')

## 2. Side-by-Side Comparison Table

In [ ]:
# Build comparison table
metric_names = [
    ('Detection Rate (Recall)', 'detection_rate_recall'),
    ('Precision', 'precision'),
    ('F1 Score', 'f1_score'),
    ('False Positive Rate', 'false_positive_rate'),
    ('Avg Line Accuracy', 'avg_line_accuracy'),
]

# Header
header = f"{'Metric':<25}"
for m in models:
    short = m.split('/')[-1][:20]
    header += f" | {short:>20}"
print(header)
print('-' * len(header))

for label, key in metric_names:
    row = f"{label:<25}"
    for m in models:
        val = model_metrics[m][key]
        row += f" | {val:>19.1%}"
    print(row)

# Latency & cost rows
print()
row = f"{'Avg Latency (ms)':<25}"
for m in models:
    val = model_metrics[m]['latency']['avg_total_ms']
    row += f" | {val:>18.0f}ms"
print(row)

row = f"{'Avg Tokens/Snippet':<25}"
for m in models:
    val = model_metrics[m]['tokens']['avg_tokens_per_snippet']
    row += f" | {val:>20.0f}"
print(row)

row = f"{'Cost/Review (USD)':<25}"
for m in models:
    val = model_metrics[m]['cost_estimate']['cost_per_review_usd']
    row += f" | ${val:>18.6f}"
print(row)

row = f"{'Total Cost (USD)':<25}"
for m in models:
    val = model_metrics[m]['cost_estimate']['total_cost_usd']
    row += f" | ${val:>18.4f}"
print(row)

## 3. Radar Chart — Model Comparison Across Dimensions

In [ ]:
if HAS_MPL and HAS_NP:
    dimensions = ['Recall', 'Precision', 'F1', 'Line Acc.', 'Low FPR']
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    angles = np.linspace(0, 2 * np.pi, len(dimensions), endpoint=False).tolist()
    angles += angles[:1]  # close the polygon
    
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
    
    for idx, model_name in enumerate(models):
        m = model_metrics[model_name]
        values = [
            m['detection_rate_recall'],
            m['precision'],
            m['f1_score'],
            m['avg_line_accuracy'],
            1.0 - m['false_positive_rate'],  # invert so higher = better
        ]
        values += values[:1]  # close polygon
        
        short = model_name.split('/')[-1][:25]
        color = colors[idx % len(colors)]
        ax.plot(angles, values, 'o-', linewidth=2, label=short, color=color)
        ax.fill(angles, values, alpha=0.1, color=color)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(dimensions, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_title('Model Comparison Radar', fontsize=14, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()
else:
    print('Radar chart requires matplotlib and numpy')

## 4. Cost vs Accuracy Tradeoff

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 7))
    
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
    
    for idx, model_name in enumerate(models):
        m = model_metrics[model_name]
        cost = m['cost_estimate']['cost_per_review_usd'] * 1000  # cost in millicents
        f1 = m['f1_score']
        short = model_name.split('/')[-1][:25]
        color = colors[idx % len(colors)]
        
        ax.scatter(cost, f1, s=200, color=color, zorder=5, edgecolors='white', linewidths=2)
        ax.annotate(short, (cost, f1), textcoords='offset points',
                    xytext=(10, 10), fontsize=11, color=color, fontweight='bold')
    
    ax.set_xlabel('Cost per Review (USD × 1000)', fontsize=12)
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title('Cost vs Accuracy Tradeoff', fontsize=14)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Chart requires matplotlib')

## 5. Error Analysis — Missed Vulnerabilities by Model

In [ ]:
# For each model, show which categories were missed
print('Missed vulnerability categories per model:\n')

for model_name in models:
    m = model_metrics[model_name]
    short = model_name.split('/')[-1][:25]
    print(f'=== {short} ===')
    cats = m['category_breakdown']
    missed_cats = {c: s for c, s in cats.items() if s['recall'] < 1.0}
    if not missed_cats:
        print('  All categories detected at 100%!')
    else:
        for cat, stats in sorted(missed_cats.items(), key=lambda x: x[1]['recall']):
            missed = stats['total'] - stats['detected']
            print(f"  {cat:<30} recall={stats['recall']:.0%}  (missed {missed}/{stats['total']})")
    print()

# Heatmap if matplotlib available
if HAS_MPL and HAS_NP and len(models) > 1:
    all_cats = set()
    for m in model_metrics.values():
        all_cats.update(m['category_breakdown'].keys())
    all_cats = sorted(all_cats)
    
    matrix = []
    for model_name in models:
        row = []
        cats = model_metrics[model_name]['category_breakdown']
        for cat in all_cats:
            row.append(cats.get(cat, {}).get('recall', 0.0))
        matrix.append(row)
    
    matrix = np.array(matrix)
    fig, ax = plt.subplots(figsize=(14, max(4, len(models) * 1.5)))
    im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    
    short_models = [m.split('/')[-1][:20] for m in models]
    ax.set_xticks(range(len(all_cats)))
    ax.set_xticklabels(all_cats, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(models)))
    ax.set_yticklabels(short_models, fontsize=11)
    
    # Annotate cells
    for i in range(len(models)):
        for j in range(len(all_cats)):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f'{val:.0%}', ha='center', va='center', color=color, fontsize=9)
    
    plt.colorbar(im, label='Recall')
    ax.set_title('Detection Rate by Category × Model', fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Latency Comparison

In [ ]:
print(f"{'Model':<30} {'Avg Summary':>12} {'Avg Inline':>12} {'Avg Total':>12}")
print('-' * 68)
for model_name in models:
    lat = model_metrics[model_name]['latency']
    short = model_name.split('/')[-1][:28]
    print(f"{short:<30} {lat['avg_summary_ms']:>10.0f}ms {lat['avg_inline_ms']:>10.0f}ms {lat['avg_total_ms']:>10.0f}ms")

if HAS_MPL and len(models) > 1:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    short_models = [m.split('/')[-1][:20] for m in models]
    summary_lats = [model_metrics[m]['latency']['avg_summary_ms'] for m in models]
    inline_lats = [model_metrics[m]['latency']['avg_inline_ms'] for m in models]
    
    x = range(len(models))
    width = 0.35
    ax.bar([i - width/2 for i in x], summary_lats, width, label='Summary', color='#3498db', alpha=0.8)
    ax.bar([i + width/2 for i in x], inline_lats, width, label='Inline', color='#e74c3c', alpha=0.8)
    
    ax.set_xlabel('Model')
    ax.set_ylabel('Latency (ms)')
    ax.set_title('Average Latency by Model')
    ax.set_xticks(x)
    ax.set_xticklabels(short_models)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 7. Final Recommendation

In [ ]:
# Score each model: weighted combination of key metrics
# Weights: F1 (40%), Recall (25%), Precision (15%), Line Accuracy (10%), Low Cost (10%)
scores = {}
max_cost = max(model_metrics[m]['cost_estimate']['cost_per_review_usd'] for m in models) or 0.001

for model_name in models:
    m = model_metrics[model_name]
    cost_norm = 1.0 - (m['cost_estimate']['cost_per_review_usd'] / max_cost) if max_cost > 0 else 1.0
    
    score = (
        0.40 * m['f1_score'] +
        0.25 * m['detection_rate_recall'] +
        0.15 * m['precision'] +
        0.10 * m['avg_line_accuracy'] +
        0.10 * cost_norm
    )
    scores[model_name] = score

# Rank
ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

print('=' * 60)
print('MODEL RANKING (weighted composite score)')
print('=' * 60)
print(f"{'Rank':<6} {'Model':<35} {'Score':>8}")
print('-' * 52)
for rank, (model_name, score) in enumerate(ranked, 1):
    short = model_name.split('/')[-1][:33]
    medal = ['🥇', '🥈', '🥉'][rank-1] if rank <= 3 else '  '
    print(f"{medal} {rank:<4} {short:<35} {score:>7.1%}")

winner = ranked[0][0]
w = model_metrics[winner]
print(f"\n{'='*60}")
print(f"RECOMMENDED MODEL: {winner}")
print(f"{'='*60}")
print(f"  F1 Score:    {w['f1_score']:.1%}")
print(f"  Recall:      {w['detection_rate_recall']:.1%}")
print(f"  Precision:   {w['precision']:.1%}")
print(f"  Cost/Review: ${w['cost_estimate']['cost_per_review_usd']:.6f}")
print(f"  Latency:     {w['latency']['avg_total_ms']:.0f}ms avg")